In [1]:
#import statements

import numpy as np
import pandas as pd
from glob import glob
from datetime import date, timedelta
import datetime
import math

In [2]:
#list out data files

fils = sorted(glob('Dataset/*.csv'))
fils

['Dataset/raw_data01.csv',
 'Dataset/raw_data02.csv',
 'Dataset/raw_data03.csv',
 'Dataset/raw_data04.csv',
 'Dataset/raw_data05.csv',
 'Dataset/raw_data06.csv',
 'Dataset/raw_data07.csv',
 'Dataset/raw_data08.csv',
 'Dataset/raw_data09.csv',
 'Dataset/raw_data10.csv',
 'Dataset/raw_data11.csv',
 'Dataset/raw_data12.csv',
 'Dataset/raw_data13.csv',
 'Dataset/raw_data14.csv',
 'Dataset/raw_data15.csv',
 'Dataset/raw_data16.csv',
 'Dataset/raw_data17.csv',
 'Dataset/raw_data18.csv',
 'Dataset/raw_data19.csv']

In [3]:
#frame dataframe from data files

case_time = pd.read_csv('Dataset/Extd/case_time_series.csv')
state_wise = pd.read_csv('Dataset/Extd/state_wise_daily.csv')
states = pd.read_csv('Dataset/Extd/states.csv')
df = pd.concat((pd.read_csv(file).assign(filename = file)
          for file in fils),ignore_index = True)
df

/home/hoozeeloogaan/Programs/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:7: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  import sys


,Age Bracket,Backup Notes,Contracted from which Patient (Suspected),Current Status,Date Announced,Detected City,Detected District,Detected State,Entry_ID,Estimated Onset Date,...,Patient Number,Source_1,Source_2,Source_3,State Patient Number,State code,Status Change Date,Type of transmission,filename,months
0,20,Student from Wuhan,NaN,Recovered,30/01/2020,Thrissur,Thrissur,Kerala,NaN,NaN,...,1.0,https://twitter.com/vijayanpinarayi/status/122...,https://weather.com/en-IN/india/news/news/2020...,NaN,KL-TS-P1,KL,14/02/2020,Imported,Dataset/raw_data01.csv,NaN
1,NaN,Student from Wuhan,NaN,Recovered,02/02/2020,Alappuzha,Alappuzha,Kerala,NaN,NaN,...,2.0,https://www.indiatoday.in/india/story/kerala-r...,https://weather.com/en-IN/india/news/news/2020...,NaN,KL-AL-P1,KL,14/02/2020,Imported,Dataset/raw_data01.csv,NaN
2,NaN,Student from Wuhan,NaN,Recovered,03/02/2020,Kasaragod,Kasaragod,Kerala,NaN,NaN,...,3.0,https://www.indiatoday.in/india/story/kerala-n...,https://twitter.com/ANI/status/122422148580539...,https://weather.com/en-IN/india/news/news/2020...,KL-KS-P1,KL,14/02/2020,Imported,Dataset/raw_data01.csv,NaN
3,45,Travel history to Italy and Austria,NaN,Recovered,02/03/2020,East Delhi (Mayur Vihar),East Delhi,Delhi,NaN,NaN,...,4.0,https://www.indiatoday.in/india/story/not-a-ja...,https://economictimes.indiatimes.com/news/poli...,NaN,DL-P1,DL,15/03/2020,Imported,Dataset/raw_data01.csv,NaN
4,24,"Travel history to Dubai, Singapore contact",NaN,Recovered,02/03/2020,Hyderabad,Hyderabad,Telangana,NaN,NaN,...,5.0,https://www.deccanherald.com/national/south/qu...,https://www.indiatoday.in/india/story/coronavi...,https://www.thehindu.com/news/national/coronav...,TS-P1,TG,02/03/2020,Imported,Dataset/raw_data01.csv,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427584,NaN,NaN,NaN,Recovered,28/11/2020,NaN,Leh,Ladakh,398941.0,NaN,...,NaN,http://covid.ladakh.gov.in/#dataInsights,NaN,NaN,NaN,LA,NaN,NaN,Dataset/raw_data19.csv,NaN
427585,NaN,NaN,NaN,Recovered,28/11/2020,NaN,Kargil,Ladakh,398942.0,NaN,...,NaN,http://covid.ladakh.gov.in/#dataInsights,NaN,NaN,NaN,LA,NaN,NaN,Dataset/raw_data19.csv,NaN
427586,NaN,NaN,NaN,Deceased,28/11/2020,NaN,Leh,Ladakh,398943.0,NaN,...,NaN,http://covid.ladakh.gov.in/#dataInsights,NaN,NaN,NaN,LA,NaN,NaN,Dataset/raw_data19.csv,NaN
427587,NaN,NaN,NaN,Recovered,28/11/2020,NaN,Ahmedabad,Gujarat,398944.0,NaN,...,NaN,https://gujcovid19.gujarat.gov.in/,NaN,NaN,NaN,GJ,NaN,NaN,Dataset/raw_data19.csv,NaN


In [4]:
#extract needed features from the data and add primary key

df = df[["Date Announced","Current Status","Detected District","Detected State","Age Bracket"]]

In [5]:
#removing all rows with age value NaN

data = df.dropna()
data_n = df.dropna(subset=['Date Announced','Current Status','Detected District'])
data = data.rename(columns = { 'Date Announced':'Date_Announced','Current Status':'Current_Status','Detected District':'Detected_District','Detected State':'Detected_State','Age Bracket':'Age_Bracket' })
Patient_id = list(range(len(data)))
data['Patient_id'] = Patient_id
data = data[data['Current_Status'] != 'Migrated_Other']
data['Detected_District'] = data['Detected_District'].str.lower()
data['Detected_State'] = data['Detected_State'].str.lower()

In [ ]:
#add age bitmap to the dataframe

newlist = [0]*len(data)
data['Age_0_15'] = newlist
data['Age_15_30'] = newlist
data['Age_30_45'] = newlist
data['Age_45_60'] = newlist
data['Age_60'] = newlist

for i in data.index:
    age = (data.loc[i,['Age_Bracket']])
    x = age.to_string()
    if len(x) >= 18:
        print(x)
        age = x.split('-')[0][-2:]
        age = math.ceil(float(age))
    age = int(age)
    if age > 0 and age <= 15:
        data.loc[i,['Age_0_15']] = 1
    elif age > 15 and age <= 30:
        data.loc[i,['Age_15_30']] = 1
    elif age > 30 and age <= 45:
        data.loc[i,['Age_30_45']] = 1
    elif age > 45 and age <= 60:
        data.loc[i,['Age_45_60']] = 1
    else:
        data.loc[i,['Age_60']] = 1
        
data

In [7]:
#district wise segregation

state_wise_filter = data[data['Detected_State'] == 'karnataka']
state_wise_filter

,Date_Announced,Current_Status,Detected_District,Detected_State,Age_Bracket,Patient_id
41,09/03/2020,Recovered,bengaluru urban,karnataka,46,33
50,10/03/2020,Recovered,bengaluru urban,karnataka,40,36
51,10/03/2020,Recovered,bengaluru urban,karnataka,47,37
52,10/03/2020,Recovered,bengaluru urban,karnataka,13,38
73,12/03/2020,Recovered,bengaluru urban,karnataka,26,49
...,...,...,...,...,...,...
299041,05/09/2020,Deceased,bengaluru urban,karnataka,65,113881
299042,05/09/2020,Deceased,hassan,karnataka,60,113882
299043,05/09/2020,Deceased,hassan,karnataka,66,113883
299044,05/09/2020,Deceased,ramanagara,karnataka,52,113884


In [8]:
district_list = state_wise_filter.Detected_District.unique()
(district_list)

district_data = dict()
for districts in district_list:
    district_data.update({districts:0})
district_data

{'bengaluru urban': 0,
 'kalaburagi': 0,
 'kodagu': 0,
 'chikkaballapura': 0,
 'mysuru': 0,
 'dharwad': 0,
 'uttara kannada': 0,
 'other state': 0,
 'udupi': 0,
 'chitradurga': 0,
 'dakshina kannada': 0,
 'tumakuru': 0,
 'davanagere': 0,
 'ballari': 0,
 'bidar': 0,
 'bagalkote': 0,
 'belagavi': 0,
 'bengaluru rural': 0,
 'gadag': 0,
 'mandya': 0,
 'vijayapura': 0,
 'haveri': 0,
 'shivamogga': 0,
 'yadgir': 0,
 'hassan': 0,
 'kolar': 0,
 'raichur': 0,
 'koppal': 0,
 'chikkamagaluru': 0,
 'ramanagara': 0,
 'chamarajanagara': 0}

In [9]:
#calander like dataframe

sdate = date(2020, 1, 1)
edate = date(2020, 12, 31)

delta = edate - sdate
dates = []

for i in range(delta.days + 1):
    day = sdate + timedelta(days=i)
    dates.append(day.strftime('%d/%m/%Y'))

In [10]:
#segregate dataset into recovered/deceased/hospitalized

recovered_data = data[data['Current_Status'] == 'Recovered']
deceased_data = data[data['Current_Status'] == 'Deceased']
hospitalized_data = data[data['Current_Status'] == 'Hospitalized']

In [11]:
#district-wise calender dataframe

for district in district_data:
    cal_dat = pd.DataFrame(columns=['Date_Announced','Total_Infections','Total_Recoveries','Total_Deceased','District','State'])
    cal_dat['Date_Announced'] = dates
    cal_dat.set_index('Date_Announced',inplace=True)
    
    buff_list = pd.DataFrame(hospitalized_data.loc[data['Detected_District']==district])
    grouped =  (buff_list.groupby('Date_Announced').groups)
    for i,j in grouped.items():
        grouped[i] = len(list(j))    

    for i in cal_dat.index:
        for j,k in grouped.items():
            if i == j:
                x = (cal_dat.loc[i,['Total_Infections']])
                if np.isnan(x['Total_Infections']):
                    cal_dat.loc[i,['Total_Infections']] = k
                else:
                    cal_dat.loc[i,['Total_Infections']] = x['Total_Infections'] + k
        cal_dat.loc[i,['District']] = district
        cal_dat.loc[i,['State']] = 'karnataka'
        
    
    buff_list = pd.DataFrame(recovered_data.loc[data['Detected_District']==district])
    grouped =  (buff_list.groupby('Date_Announced').groups)
    for i,j in grouped.items():
        grouped[i] = len(list(j))    

    for i in cal_dat.index:
        for j,k in grouped.items():
            if i == j:
                x = (cal_dat.loc[i,['Total_Recoveries']])
                if np.isnan(x['Total_Recoveries']):
                    cal_dat.loc[i,['Total_Recoveries']] = k
                else:
                    cal_dat.loc[i,['Total_Recoveries']] = x['Total_Recoveries'] + k
        
    
    buff_list = pd.DataFrame(deceased_data.loc[data['Detected_District']==district])
    grouped =  (buff_list.groupby('Date_Announced').groups)
    for i,j in grouped.items():
        grouped[i] = len(list(j))    

    for i in cal_dat.index:
        for j,k in grouped.items():
            if i == j:
                x = (cal_dat.loc[i,['Total_Deceased']])
                if np.isnan(x['Total_Deceased']):
                    cal_dat.loc[i,['Total_Deceased']] = k
                else:
                    cal_dat.loc[i,['Total_Deceased']] = x['Total_Deceased'] + k
        
    cal_dat['Total_Recoveries'] = cal_dat['Total_Recoveries'].replace(np.nan, 'DNA')
    cal_dat['Total_Deceased'] = cal_dat['Total_Deceased'].replace(np.nan, 'DNA') 
    cal_dat['Total_Infections'] = cal_dat['Total_Infections'].replace(np.nan, 'DNA') 
    district_data[district] = cal_dat


In [12]:
#specific o/p

pd.set_option('display.max_rows',466)
district_data['bidar']

,Total_Infections,Total_Recoveries,Total_Deceased,District,State
Date_Announced,,,,,
01/01/2020,DNA,DNA,DNA,bidar,karnataka
02/01/2020,DNA,DNA,DNA,bidar,karnataka
03/01/2020,DNA,DNA,DNA,bidar,karnataka
04/01/2020,DNA,DNA,DNA,bidar,karnataka
05/01/2020,DNA,DNA,DNA,bidar,karnataka
06/01/2020,DNA,DNA,DNA,bidar,karnataka
07/01/2020,DNA,DNA,DNA,bidar,karnataka
08/01/2020,DNA,DNA,DNA,bidar,karnataka
09/01/2020,DNA,DNA,DNA,bidar,karnataka


In [7]:
#state wise segregation

state_list = data.Detected_State.unique()

state_data = dict()
for states in state_list:
    state_data.update({states:0})
state_data

{'Kerala': 0,
 'Delhi': 0,
 'Telangana': 0,
 'Rajasthan': 0,
 'Haryana': 0,
 'Uttar Pradesh': 0,
 'Ladakh': 0,
 'Tamil Nadu': 0,
 'Jammu and Kashmir': 0,
 'Karnataka': 0,
 'Maharashtra': 0,
 'Andhra Pradesh': 0,
 'Odisha': 0,
 'Puducherry': 0,
 'West Bengal': 0,
 'Chandigarh': 0,
 'Chhattisgarh': 0,
 'Punjab': 0,
 'Gujarat': 0,
 'Himachal Pradesh': 0,
 'Madhya Pradesh': 0,
 'Bihar': 0,
 'Uttarakhand': 0,
 'Manipur': 0,
 'Mizoram': 0,
 'Goa': 0,
 'Andaman and Nicobar Islands': 0,
 'Jharkhand': 0,
 'Assam': 0,
 'Tripura': 0,
 'Meghalaya': 0,
 'Dadra and Nagar Haveli and Daman and Diu': 0,
 'Sikkim': 0,
 'Arunachal Pradesh': 0}

In [9]:
#state-wise calender dataframe 

for state in state_data:
    cal_dat = pd.DataFrame(columns=['Date_Announced','Total_Infections','Total_Recoveries','Total_Deceased','State'])
    cal_dat['Date_Announced'] = dates
    cal_dat.set_index('Date_Announced',inplace=True)
    
    buff_list = pd.DataFrame(hospitalized_data.loc[data['Detected_State']==state])
    grouped =  (buff_list.groupby('Date_Announced').groups)
    for i,j in grouped.items():
        grouped[i] = len(list(j))    

    for i in cal_dat.index:
        for j,k in grouped.items():
            if i == j:
                x = (cal_dat.loc[i,['Total_Infections']])
                if np.isnan(x['Total_Infections']):
                    cal_dat.loc[i,['Total_Infections']] = k
                else:
                    cal_dat.loc[i,['Total_Infections']] = x['Total_Infections'] + k
        cal_dat.loc[i,['State']] = state
        
    
    buff_list = pd.DataFrame(recovered_data.loc[data['Detected_State']==state])
    grouped =  (buff_list.groupby('Date_Announced').groups)
    for i,j in grouped.items():
        grouped[i] = len(list(j))    

    for i in cal_dat.index:
        for j,k in grouped.items():
            if i == j:
                x = (cal_dat.loc[i,['Total_Recoveries']])
                if np.isnan(x['Total_Recoveries']):
                    cal_dat.loc[i,['Total_Recoveries']] = k
                else:
                    cal_dat.loc[i,['Total_Recoveries']] = x['Total_Recoveries'] + k
        
    
    buff_list = pd.DataFrame(deceased_data.loc[data['Detected_State']==state])
    grouped =  (buff_list.groupby('Date_Announced').groups)
    for i,j in grouped.items():
        grouped[i] = len(list(j))    

    for i in cal_dat.index:
        for j,k in grouped.items():
            if i == j:
                x = (cal_dat.loc[i,['Total_Deceased']])
                if np.isnan(x['Total_Deceased']):
                    cal_dat.loc[i,['Total_Deceased']] = k
                else:
                    cal_dat.loc[i,['Total_Deceased']] = x['Total_Deceased'] + k
        
            
    cal_dat['Total_Recoveries'] = cal_dat['Total_Recoveries'].replace(np.nan, 'DNA')
    cal_dat['Total_Deceased'] = cal_dat['Total_Deceased'].replace(np.nan, 'DNA') 
    cal_dat['Total_Infections'] = cal_dat['Total_Infections'].replace(np.nan, 'DNA') 
    state_data[state] = cal_dat


In [10]:
state_data

{'Kerala':                Total_Infections Total_Recoveries Total_Deceased   State
 Date_Announced                                                         
 01/01/2020                  NaN              NaN            NaN  Kerala
 02/01/2020                  NaN              NaN            NaN  Kerala
 03/01/2020                  NaN              NaN            NaN  Kerala
 04/01/2020                  NaN              NaN            NaN  Kerala
 05/01/2020                  NaN              NaN            NaN  Kerala
 ...                         ...              ...            ...     ...
 27/12/2020                  NaN              NaN            NaN  Kerala
 28/12/2020                  NaN              NaN            NaN  Kerala
 29/12/2020                  NaN              NaN            NaN  Kerala
 30/12/2020                  NaN              NaN            NaN  Kerala
 31/12/2020                  NaN              NaN            NaN  Kerala
 
 [366 rows x 4 columns],
 'Delhi':     

In [12]:
pd.set_option('display.max_rows',466)
state_data['Karnataka']

,Total_Infections,Total_Recoveries,Total_Deceased,State
Date_Announced,,,,
01/01/2020,NaN,NaN,NaN,Karnataka
02/01/2020,NaN,NaN,NaN,Karnataka
03/01/2020,NaN,NaN,NaN,Karnataka
04/01/2020,NaN,NaN,NaN,Karnataka
05/01/2020,NaN,NaN,NaN,Karnataka
06/01/2020,NaN,NaN,NaN,Karnataka
07/01/2020,NaN,NaN,NaN,Karnataka
08/01/2020,NaN,NaN,NaN,Karnataka
09/01/2020,NaN,NaN,NaN,Karnataka


In [ ]:
-----------------------------------------------------**-----------------------------------------------------------

In [5]:
#Other datasets

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300911 entries, 0 to 300910
Data columns (total 23 columns):
Age Bracket                                  113961 non-null object
Backup Notes                                 361 non-null object
Contracted from which Patient (Suspected)    1938 non-null object
Current Status                               300905 non-null object
Date Announced                               300909 non-null object
Detected City                                13841 non-null object
Detected District                            291105 non-null object
Detected State                               300898 non-null object
Entry_ID                                     272726 non-null float64
Estimated Onset Date                         0 non-null float64
Gender                                       116592 non-null object
Nationality                                  1554 non-null object
Notes                                        114332 non-null object
Num Cases       

In [6]:
pd.set_option('max_rows', None)
case_time

,Date,Daily Confirmed,Total Confirmed,Daily Recovered,Total Recovered,Daily Deceased,Total Deceased
0,30 January,1,1,0,0,0,0
1,31 January,0,1,0,0,0,0
2,01 February,0,1,0,0,0,0
3,02 February,1,2,0,0,0,0
4,03 February,1,3,0,0,0,0
5,04 February,0,3,0,0,0,0
6,05 February,0,3,0,0,0,0
7,06 February,0,3,0,0,0,0
8,07 February,0,3,0,0,0,0
9,08 February,0,3,0,0,0,0


,Date,Status,TT,AN,AP,AR,AS,BR,CH,CT,...,PB,RJ,SK,TN,TG,TR,UP,UT,WB,UN
0,14-Mar-20,Confirmed,81,0,1,0,0,0,0,0,...,1,3,0,1,1,0,12,0,0,0
1,14-Mar-20,Recovered,9,0,0,0,0,0,0,0,...,0,1,0,0,0,0,4,0,0,0
2,14-Mar-20,Deceased,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,15-Mar-20,Confirmed,27,0,0,0,0,0,0,0,...,0,1,0,0,2,0,1,0,0,0
4,15-Mar-20,Recovered,4,0,0,0,0,0,0,0,...,0,2,0,0,1,0,0,0,0,0
5,15-Mar-20,Deceased,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,16-Mar-20,Confirmed,15,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
7,16-Mar-20,Recovered,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
8,16-Mar-20,Deceased,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,17-Mar-20,Confirmed,11,0,0,0,0,0,0,0,...,0,0,0,0,1,0,2,0,1,0


,Date,State,Confirmed,Recovered,Deceased,Other,Tested
0,2020-01-30,Kerala,1,0,0,0,NaN
1,2020-01-30,India,1,0,0,0,NaN
2,2020-02-02,Kerala,2,0,0,0,NaN
3,2020-02-02,India,2,0,0,0,NaN
4,2020-02-03,Kerala,3,0,0,0,NaN
5,2020-02-03,India,3,0,0,0,NaN
6,2020-02-14,Kerala,3,3,0,0,NaN
7,2020-02-14,India,3,3,0,0,NaN
8,2020-03-02,Delhi,1,0,0,0,NaN
9,2020-03-02,Kerala,3,3,0,0,NaN


Age_Bracket    28-35
Age_Bracket    28-35
Age_Bracket    28-35
Age_Bracket    28-35
Age_Bracket    1.5
Age_Bracket    0.4
Age_Bracket    0.3
Age_Bracket    1.5
Age_Bracket    2.5
Age_Bracket    0.2
Age_Bracket    0.2
Age_Bracket    0.7
Age_Bracket    0.3
Age_Bracket    29.6
Age_Bracket    0.1
Age_Bracket    0.1
Age_Bracket    3.5
Age_Bracket    0.1
Age_Bracket    2.5
Age_Bracket    3.5
Age_Bracket    0.5
Age_Bracket    0.2
Age_Bracket    2.5
Age_Bracket    0.9
Age_Bracket    1.5
Age_Bracket    1.5
Age_Bracket    1.5
Age_Bracket    0.5
Age_Bracket    0.25
Age_Bracket    0.1
Age_Bracket    0.1
Age_Bracket    1.5
Age_Bracket    0.9
Age_Bracket    0.1
Age_Bracket    0.1
Age_Bracket    2.5
Age_Bracket    3.5
Age_Bracket    1.5
Age_Bracket    0.7
Age_Bracket    0.1
Age_Bracket    0.5
Age_Bracket    0.1
Age_Bracket    0.1
Age_Bracket    1.5
Age_Bracket    2.5
Age_Bracket    0.1
Age_Bracket    54.9
Age_Bracket    0.7
Age_Bracket    0.1
Age_Bracket    0.1
Age_Bracket    0.6
Age_Bracket    0.3
A

,Date_Announced,Current_Status,Detected_District,Detected_State,Age_Bracket,Patient_id,Age_0_15,Age_15_30,Age_30_45,Age_45_60,Age_60
0,30/01/2020,Recovered,thrissur,kerala,20,0,0,1,0,0,0
3,02/03/2020,Recovered,east delhi,delhi,45,1,0,0,1,0,0
4,02/03/2020,Recovered,hyderabad,telangana,24,2,0,1,0,0,0
5,03/03/2020,Recovered,italians,rajasthan,69,3,0,0,0,0,1
6,04/03/2020,Recovered,italians,haryana,55,4,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
427574,28/11/2020,Deceased,imphal west,manipur,62,116642,0,0,0,0,1
427575,28/11/2020,Deceased,imphal west,manipur,69,116643,0,0,0,0,1
427576,28/11/2020,Deceased,imphal west,manipur,62,116644,0,0,0,0,1
427577,28/11/2020,Deceased,imphal east,manipur,80,116645,0,0,0,0,1


In [113]:
for i in test_data.index:
    age = (test_data.loc[i,['Age_Bracket']])
    x = age.to_string()
    if len(x) >= 18:
        age = x.split('-')[0][-2:]
    age = int(age)
    if age > 0 and age <= 15:
        test_data.loc[i,['Age_0_15']] = 1
    elif age > 15 and age <= 30:
        test_data.loc[i,['Age_15_30']] = 1
    elif age > 30 and age <= 45:
        test_data.loc[i,['Age_30_45']] = 1
    elif age > 45 and age <= 60:
        test_data.loc[i,['Age_45_60']] = 1
    else:
        test_data.loc[i,['Age_60']] = 1
test_data

NameError: name 'ceil' is not defined

In [9]:
test_data = data.loc[1:50]

In [13]:
test_data

,Date_Announced,Current_Status,Detected_District,Detected_State,Age_Bracket,Patient_id,Age_0_15,Age_15_30,Age_30_45,Age_45_60,Age_60
3,02/03/2020,Recovered,east delhi,delhi,45,1,0,0,0,0,0
4,02/03/2020,Recovered,hyderabad,telangana,24,2,0,0,0,0,0
5,03/03/2020,Recovered,italians,rajasthan,69,3,0,0,0,0,0
6,04/03/2020,Recovered,italians,haryana,55,4,0,0,0,0,0
7,04/03/2020,Recovered,italians,haryana,55,5,0,0,0,0,0
8,04/03/2020,Recovered,italians,haryana,55,6,0,0,0,0,0
9,04/03/2020,Recovered,italians,haryana,55,7,0,0,0,0,0
10,04/03/2020,Recovered,italians,haryana,55,8,0,0,0,0,0
11,04/03/2020,Recovered,italians,haryana,55,9,0,0,0,0,0
12,04/03/2020,Recovered,italians,haryana,55,10,0,0,0,0,0


In [14]:
test_data.loc[20,['Age_Bracket']] = '20-30'

/home/hoozeeloogaan/Programs/anaconda3/lib/python3.7/site-packages/pandas/core/indexing.py:494: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[item] = s


In [99]:
test_data = data.loc[1:50]

TypeError: '>' not supported between instances of 'str' and 'int'

In [12]:
newlist = [0]*len(test_data)
test_data['Age_0_15'] = newlist
test_data['Age_15_30'] = newlist
test_data['Age_30_45'] = newlist
test_data['Age_45_60'] = newlist
test_data['Age_60'] = newlist


/home/hoozeeloogaan/Programs/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/home/hoozeeloogaan/Programs/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until
/home/hoozeeloogaan/Programs/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of

In [ ]:
for i in a:
    b.append(str(a))

In [24]:
b = "10"
print(b.isnumeric())
print(type(b))

True
<class 'str'>


In [102]:
data

,Date_Announced,Current_Status,Detected_District,Detected_State,Age_Bracket,Patient_id,Age_0_15,Age_15_30,Age_30_45,Age_45_60,Age_60
0,30/01/2020,Recovered,thrissur,kerala,20,0,0,1,0,0,0
3,02/03/2020,Recovered,east delhi,delhi,45,1,0,0,1,0,0
4,02/03/2020,Recovered,hyderabad,telangana,24,2,0,1,0,0,0
5,03/03/2020,Recovered,italians,rajasthan,69,3,0,0,0,0,1
6,04/03/2020,Recovered,italians,haryana,55,4,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
427574,28/11/2020,Deceased,imphal west,manipur,62,116643,0,0,0,0,0
427575,28/11/2020,Deceased,imphal west,manipur,69,116644,0,0,0,0,0
427576,28/11/2020,Deceased,imphal west,manipur,62,116645,0,0,0,0,0
427577,28/11/2020,Deceased,imphal east,manipur,80,116646,0,0,0,0,0


In [112]:
x = '.5'
float(x)

0.5

In [10]:
df[df['Age Bracket'] == 's']

,Age Bracket,Backup Notes,Contracted from which Patient (Suspected),Current Status,Date Announced,Detected City,Detected District,Detected State,Entry_ID,Estimated Onset Date,...,Patient Number,Source_1,Source_2,Source_3,State Patient Number,State code,Status Change Date,Type of transmission,filename,months
240223,s,NaN,NaN,Deceased,02/08/2020,NaN,Srikakulam,Andhra Pradesh,211982.0,NaN,...,NaN,https://twitter.com/ArogyaAndhra/status/128991...,NaN,NaN,NaN,AP,NaN,NaN,Dataset/raw_data12.csv,NaN
